# SmartBag — MLP / TinyML
Mesmas seis features, classes e rodadas da RF. Rede pequena: 6 → 8 → 1. Normalização fora da rede, repetida no ESP32.

## 1. Pacotes

In [ ]:
%pip -q install pandas numpy matplotlib scikit-learn tensorflow==2.20.0


In [ ]:
import json, hashlib, zipfile
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
tf.keras.utils.set_random_seed(42)


## 2. Abrir e conferir o CSV


In [ ]:
from google.colab import files
arquivos = files.upload()
ARQUIVO = next(iter(arquivos))
df = pd.read_csv(ARQUIVO).sort_values(["rodada", "situacao", "timestamp"]).reset_index(drop=True)
FEATURES = ["temperatura", "umidade", "delta_distancia", "luz", "mov_max", "incl_max"]
CLASSES = ["ENTREGA_OK", "REVISAR_ENTREGA"]
X = df[FEATURES].astype(np.float32)
y = df["target"].map({nome: i for i, nome in enumerate(CLASSES)})
assert y.notna().all(), "Confira os targets do CSV."
assert df["device"].nunique() == 1, "Use uma execução de uma equipe."
display(pd.crosstab(df["rodada"], df["target"]))


## 3. Separar por rodada
A última rodada fica no teste, como no app17-7. RF e MLP usam o mesmo CSV e a mesma divisão.


In [ ]:
rodada_teste = df["rodada"].max()
indices_treino = df.index[df["rodada"] != rodada_teste]
indices_teste = df.index[df["rodada"] == rodada_teste]
X_treino, X_teste = X.loc[indices_treino], X.loc[indices_teste]
y_treino, y_teste = y.loc[indices_treino], y.loc[indices_teste]
rodadas_treino = sorted(df.loc[indices_treino, "rodada"].unique().tolist())
rodadas_teste = [int(rodada_teste)]
assert set(y_treino) == set(y_teste) == {0, 1}, "Colete pelo menos duas rodadas completas com as duas classes."
print("Treino:", rodadas_treino, "| Teste:", rodadas_teste)


## 4. Normalizar e treinar
Scaler ajustado somente no treino. São 50 épocas fixas; o teste não participa do fit ou de early stopping.

In [ ]:
scaler = StandardScaler().fit(X_treino)
X_treino_norm = scaler.transform(X_treino).astype(np.float32)
X_teste_norm = scaler.transform(X_teste).astype(np.float32)
modelo = tf.keras.Sequential([
    tf.keras.Input(shape=(6,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(X_treino_norm, y_treino.to_numpy(), epochs=50, batch_size=32, verbose=0)
modelo.summary()
probabilidades = modelo.predict(X_teste_norm, verbose=0).ravel()
predicoes = (probabilidades >= 0.5).astype(int)


## 5. Avaliar

In [ ]:
print("Acurácia:", accuracy_score(y_teste, predicoes))
print(classification_report(y_teste, predicoes, labels=[0, 1], target_names=CLASSES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_teste, predicoes, labels=[0, 1], display_labels=CLASSES, cmap="Blues")
plt.show()
display(pd.crosstab(df.iloc[indices_teste]["situacao"], pd.Series(predicoes, index=y_teste.index, name="predicao")))


## 6. Converter e conferir TFLite
Float32, sem quantização. A entrada já está normalizada.

In [ ]:
conversor = tf.lite.TFLiteConverter.from_keras_model(modelo)
tflite = conversor.convert()
Path("modelo_smartbag.tflite").write_bytes(tflite)
interpreter = tf.lite.Interpreter(model_content=tflite)
interpreter.allocate_tensors()
entrada = interpreter.get_input_details()[0]
saida = interpreter.get_output_details()[0]
assert list(entrada["shape"]) == [1, 6] and list(saida["shape"]) == [1, 1]
assert entrada["dtype"] == np.float32 and saida["dtype"] == np.float32
prob_tflite = []
for x in X_teste_norm:
    interpreter.set_tensor(entrada["index"], x.reshape(1, 6))
    interpreter.invoke()
    prob_tflite.append(float(interpreter.get_tensor(saida["index"])[0, 0]))
prob_tflite = np.array(prob_tflite)
np.testing.assert_allclose(prob_tflite, probabilidades, atol=1e-5, rtol=1e-5)
assert np.array_equal(prob_tflite >= 0.5, predicoes), "Divergência de classe perto de 0,5: inspecione antes de exportar."
print("Keras e TFLite conferidos. Bytes:", len(tflite))


## 7. Exportar modelo e scaler
No ESP32: normalizar uma vez, alimentar as seis entradas e decidir REVISAR_ENTREGA quando a saída for ≥ 0,5.

In [ ]:
bytes_cpp = ", ".join(f"0x{b:02x}" for b in tflite)
Path("modelo_smartbag.h").write_text(
    "#pragma once\n#include <stdint.h>\nalignas(16) const unsigned char modelo_smartbag_tflite[] = {" + bytes_cpp + "};\n",
    encoding="utf-8")
media = scaler.mean_.astype(np.float32)
escala = scaler.scale_.astype(np.float32)
media_cpp = ", ".join(f"{v:.9e}f" for v in media)
escala_cpp = ", ".join(f"{v:.9e}f" for v in escala)
header = """#pragma once
namespace Scaler {
    const float media[6] = {MEDIA};
    const float escala[6] = {ESCALA};
    void normalizar(const float entrada[6], float saida[6]) {
        for (int i = 0; i < 6; i++) saida[i] = (entrada[i] - media[i]) / escala[i];
    }
}
""".replace("MEDIA", media_cpp).replace("ESCALA", escala_cpp)
Path("AIoTSmartBagScaler.hpp").write_text(header, encoding="utf-8")
normalizado_esp = (X_teste.to_numpy(np.float32) - media) / escala
np.testing.assert_allclose(normalizado_esp, X_teste_norm, atol=1e-5, rtol=1e-5)
PACOTES = ["tensorflow", "scikit-learn", "numpy", "pandas"]
metadados = {
    "features": FEATURES, "classes": CLASSES,
    "unidades": ["C", "%", "cm", "RAW (0..4095)", "m/s2", "graus"],
    "rodadas_treino": rodadas_treino, "rodadas_teste": rodadas_teste,
    "csv_sha256": hashlib.sha256(Path(ARQUIVO).read_bytes()).hexdigest(),
    "versoes": {p: version(p) for p in PACOTES},
}
metadados.update({"media": media.tolist(), "escala": escala.tolist(), "limiar": 0.5})
Path("metadados_mlp.json").write_text(json.dumps(metadados, indent=2), encoding="utf-8")
conferencia = X_teste.copy()
conferencia["probabilidade"] = prob_tflite
conferencia["classe"] = predicoes
conferencia.to_csv("conferencia_mlp.csv", index=False)


## 8. Baixar
O app22 usa os dois headers. Confira também os resultados no ESP32: o teste local do TFLite não substitui esse ensaio.

In [ ]:
with zipfile.ZipFile("smartbag_mlp.zip", "w") as pacote:
    for nome in ["modelo_smartbag.tflite", "modelo_smartbag.h", "AIoTSmartBagScaler.hpp", "metadados_mlp.json", "conferencia_mlp.csv"]:
        pacote.write(nome)
files.download("smartbag_mlp.zip")
